In [14]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
import pickle
import os 
import warnings
warnings.filterwarnings('ignore')

In [15]:
dataset = pd.read_csv("../data/amazon_products.csv", nrows=10000)
print(f"===Dataset Loaded===")
print(f"Columns: {dataset.columns.to_list()}")

===Dataset Loaded===
Columns: ['asin', 'title', 'imgUrl', 'productURL', 'stars', 'reviews', 'price', 'listPrice', 'category_id', 'isBestSeller', 'boughtInLastMonth']


In [16]:
# Loading Content Model
from sklearn.metrics.pairwise import cosine_similarity
try:
    with open('../artifacts/tfidf_vectorizer.pkl', 'rb') as f:
        vectorizer = pickle.load(f)
    print("===Loaded TF-IDF vectorizer===")

except:
    print("Vectorizer Not found Creating new one...")
    vectorizer = TfidfVectorizer(stop_words='english', max_features=2000)

# Create product features
def create_text(row):
    features = [str(row['title']), f"category_{row['category_id']}"]
    if row['price'] > 0:
        if row['price'] < 50:
            features.append("Budget")
        elif row['price'] < 200:
            features.append("Mid-range")
        else:
            features.append("Premium")
    
    if row['isBestSeller']:
        features.append("bestseller")
    return " ".join(features)

dataset['product_text'] = dataset.apply(create_text, axis = 1)

# Create TF-IDF matrix
tfidf_matrix = vectorizer.fit_transform(dataset['product_text'])

print(f" TF-IDF matrix shape: {tfidf_matrix.shape}")
print("Contel Model is ready - using Cosine Similarity")


===Loaded TF-IDF vectorizer===
 TF-IDF matrix shape: (10000, 5000)
Contel Model is ready - using Cosine Similarity


In [17]:
# Create content function using cosine similarity
def get_content_similarity(product_idx):
    """Get content-based similarity scores using cosine_similarity"""
    content_scores = cosine_similarity(tfidf_matrix[product_idx], tfidf_matrix)[0]
    # Get top 51 indices (including itself)
    indices = content_scores.argsort()[::-1][:51]
    scores = content_scores[indices]
    return indices, scores
def recommend_content(dataset, product_name, n=5):
    """Pure content recommendation"""
    matching = dataset[dataset['title'].str.contains(product_name, case=False)]
    if len(matching) == 0:
        return f"Product '{product_name}' not found"
    
    product_idx = matching.index[0]
    indices, scores = get_content_similarity(product_idx)
    
    recommendations = []
    for i in range(1, len(indices)):
        idx = indices[i]
        recommendations.append({
            'title': dataset.iloc[idx]['title'],
            'price': dataset.iloc[idx]['price'],
            'stars': dataset.iloc[idx]['stars'],
            'similarity': round(scores[i], 3)
        })
    
    return pd.DataFrame(recommendations[:n])

print(" Content Model ready - using cosine_similarity)")
print(f"Test content recommendation: {recommend_content(dataset, 'Luggage', 3).iloc[0]['title'][:40] if len(recommend_content(dataset, 'Luggage', 3)) > 0 else 'N/A'}...")

 Content Model ready - using cosine_similarity)
Test content recommendation: Ascella X Softside Expandable Luggage wi...
